In [1]:
import dotenv

import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterstats

from rasterstats import zonal_stats
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.mask import mask

from food_security import salinity_correction, water_quality
from food_security.fao_api import FAOClient

from pathlib import Path

In [2]:
config = dotenv.dotenv_values(".env")
username = config["FAOSTAT_USERNAME"]
password = config["FAOSTAT_PASSWORD"]

fao_client = FAOClient(username=username, password=password)

In [3]:
src_dir = Path('~').expanduser() / "OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/"

In [4]:
toml_file = src_dir.parent / "salinity_correction_egypt.toml"
corrected_df = salinity_correction.generate_crop_yield_csv(
    config_path=toml_file,
    add_labor=True,
    save=False,
    fao_client=fao_client
)

2026-07-24 15:27:52,168 | WARNING  | root | config input salinity_correction.crop_production.path contains a non-existing path
2026-07-24 15:27:52,172 | WARNING  | root | config input salinity_correction.mapping.path contains a non-existing path
2026-07-24 15:27:52,829 | INFO     | food_security.salinity_correction | Starting crop yield correction for Egypt
2026-07-24 15:27:52,830 | INFO     | food_security.salinity_correction | Loaded input data. Areas=68, Crops=31, Years=2


Areas:   0%|          | 0/68 [00:00<?, ?it/s]

2026-07-24 15:28:15,393 | INFO     | food_security.salinity_correction | Created dataframe with 3226 rows
2026-07-24 15:28:15,394 | INFO     | food_security.salinity_correction | Aggregating results to department level
2026-07-24 15:28:18,110 | INFO     | food_security.salinity_correction | Crop yield correction finished successfully
/Users/hemert/projects/egypt-survey-ml/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver ESRI Shapefile does not support open option CRS
  return ogr_read(


In [5]:
relevant_crops = [
    "SummerMaize_MAIZES1 (ha)",
    "NiliMaize_MAIZEN1 (ha)",
    "Wheat_WHEAT1 (ha)",
    "LongRice_PADDY1 (ha)",
    "SugarCane_SCANE1 (ha)",
    "LongBerseem_LBSEEM1 (ha)",
    "ShortBerseem_SBSEMW1 (ha)",
]

excel_salinity_df = pd.DataFrame(
    {
        "crop_name": corrected_df["crop_name"],
        "crop_name_fao": corrected_df["crop_name_fao"],
        "area": corrected_df["area_map_name"],
        "yield": corrected_df["corrected_yield"],
        "cultivation_area (ha)": corrected_df["hectares"],
        "year": corrected_df["year"],
    }
)

excel_drought_df = pd.DataFrame(
    {
        "crop_name": corrected_df["crop_name"],
        "crop_name_fao": corrected_df["crop_name_fao"],
        "area": corrected_df["area_map_name"],
        "yield": corrected_df["yield"],
        "cultivation_area (ha)": corrected_df["hectares"],
        "year": corrected_df["year"],
    }
)
    
excel_salinity_df = excel_salinity_df[excel_salinity_df['crop_name'].isin(relevant_crops)]
excel_drought_df = excel_drought_df[excel_drought_df['crop_name'].isin(relevant_crops)]

In [6]:
excel_path = "/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt_ERF_data/data_correlation.xlsx"

def write_excel_file(df, excel_path, sheet_name, append=False):
    # Open Excel file
    try:
        # book = load_workbook(excel_path)
        with pd.ExcelWriter(
            excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
        ) as writer:
            # excel_file.book = book
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    except Exception as e:
        print(e)
        df.to_excel(excel_path, sheet_name=sheet_name, index=False)

In [7]:
command_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
conversion_df = pd.read_excel(excel_path, sheet_name='command_area')

mapping = (
    command_gdf.merge(
        conversion_df[['area_map_name', 'area_name']],
        left_on='OBJECTID',
        right_on='area_map_name'
    )
    .set_index('Name')['area_name']
)

excel_salinity_df['area'] = excel_salinity_df['area'].map(mapping)
excel_drought_df['area'] = excel_drought_df['area'].map(mapping)

In [8]:
write_excel_file(excel_salinity_df, excel_path, sheet_name="production_salinity")
write_excel_file(excel_drought_df, excel_path, sheet_name="production_drought")

In [18]:
pp_df = pd.read_excel(excel_path, sheet_name='farm_gate_price')

excel_crop_df = pd.DataFrame(
    {
        "area": excel_salinity_df["area"].unique()
    }
)

for i, row in excel_crop_df.iterrows():
    pp_total = 0
    area_name = row["area"]
    salinity_crops_row = excel_salinity_df[(excel_salinity_df['area'] == area_name) & (excel_salinity_df['year'] == 2021)]
    drought_crops_row = excel_drought_df[(excel_drought_df['area'] == area_name) & (excel_drought_df['year'] == 2021)]
    for crop_name in relevant_crops:
        salinity_crop_row = salinity_crops_row[salinity_crops_row['crop_name'] == crop_name]
        crop_name_fao = salinity_crop_row['crop_name_fao'].iloc[0]
        excel_crop_df.loc[i, f"salinity_production_{crop_name_fao}"] = salinity_crop_row['yield'].iloc[0]
        excel_crop_df.loc[i, f"salinity_cultivation_area_{crop_name_fao}"] = salinity_crop_row['cultivation_area (ha)'].iloc[0]

        drought_crop_row = drought_crops_row[drought_crops_row['crop_name'] == crop_name]
        excel_crop_df.loc[i, f"drought_production_{crop_name_fao}"] = drought_crop_row['yield'].iloc[0]
        excel_crop_df.loc[i, f"drought_cultivation_area_{crop_name_fao}"] = drought_crop_row['cultivation_area (ha)'].iloc[0]

        pp_row = pp_df[pp_df['crop_name_fao'] == crop_name_fao]
        pp_crop = salinity_crop_row['yield'].iloc[0] * pp_row['Farmgate price\n(000 EGP/ton)'].iloc[0]
        pp_total += pp_crop
    
    excel_crop_df.loc[i, 'producer_price'] = pp_total

In [21]:
excel_command_df = pd.read_excel(excel_path, sheet_name='command_unit')

excel_command_df = (
    excel_command_df
    .drop(columns=excel_crop_df.columns.difference(["area"]), errors="ignore")
    .merge(excel_crop_df, on="area", how="left")
)

for column in excel_crop_df.columns:
    if column != "area" and column != 'producer_price' and 'cultivation' not in column:
        excel_command_df[column] = excel_command_df[column] / excel_command_df['population']
    if column == "producer_price":
        excel_command_df[column] = excel_command_df[column] / excel_command_df['arable_km2']

write_excel_file(excel_command_df, excel_path, sheet_name='command_unit')